## Introduction

The purpose of this notebook is to demonstrate the usage of the `vib_frags` package for obtaining the (BLISS-)Pauli LCU of two-mode coupling vibrational Hamiltonians, as well as calculating the associated costs for QPE.

## Importing modules

In [1]:
import numpy as np
import h5py

from vib_frags import bliss_pauli as bp
from vib_frags import bliss_utils as bu
from vib_frags import tensor_utils as tu
from vib_frags import cost_utils as cu
%load_ext autoreload
%autoreload 2

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


## Importing Hamiltonian

Hamiltonians used for Trotter estimates are smaller and are stored in a different convention from the PAH Hamiltonians. That distinction is made in how the loaded tensors are transposed.

In [27]:
mol_name = 'pah1'                                                     #Can be 'pah1', 'pah2', ..., 'pah7', or 'H2S', 'CO2', 'CH2O', 'CH4' 
au_to_cm1 = 219474.6                                                  #Stored tensors are in atomic units
mc_level = 2                                                          #Mode-coupling level
nmodals = 5                                                           #Number of modals
Ham_source = 'PAH'                                                    #Can be 'Trotter' or 'PAH'. In the former case, the Hamiltonians considered are much smaller than the PAH Hamiltonians

#Loading the Hamiltonian
if Ham_source == 'Trotter':
  datafile = h5py.File(rf'Hamiltonians/Trotter_Hamiltonians/{mol_name}_loc.hdf5', 'r')
elif Ham_source == 'PAH':
  datafile = h5py.File(rf'Hamiltonians/PAH_Hamiltonians/{mol_name}/{mol_name}_{mc_level}mode/{mol_name}_VSCF_{nmodals}M_{mc_level}MC.hdf5', 'r')

#Extracting Hamiltonian terms at different mode-coupling levels
#One-mode-coupling term
H1_full = np.array(datafile['H1'][()]) * au_to_cm1     
H1 = H1_full[:, :nmodals, :nmodals]

#Two-mode-coupling term
H2_full = np.array(datafile['H2'][()]) * au_to_cm1      
if Ham_source == 'Trotter':
  H2_nonsym = np.transpose(H2_full, (0,2,4,1,3,5))                          #Permuting axes to match the convention in the paper
elif Ham_source == 'PAH':
  H2_nonsym = np.transpose(H2_full, (3,4,5,0,1,2))
H2_nonsym = H2_nonsym[:, :nmodals, :nmodals, :, :nmodals, :nmodals]       #Non symmetrized tensor
H2 = tu.symmetrize_tbt(H2_nonsym)

#Three-mode-coupling term
if Ham_source == 'PAH':
  try:
    H3_full = np.array(datafile['H3'][()]) * au_to_cm1
    H3_nonsym = np.transpose(H3_full, (6,7,8,3,4,5,0,1,2))
    H3_nonsym = H3_nonsym[:, :nmodals, :nmodals, :, :nmodals, :nmodals, :, :nmodals, :nmodals]
    H3 = tu.symmetrize_trbt(H3_nonsym)
  except KeyError:
    print ("3 mode term is missing. Setting 3 mode tensor to None.")
    H3_full, H3_nonsym, H3 = None, None, None
elif Ham_source == 'Trotter':
  H3_full, H3_nonsym, H3 = None, None, None
  mc_level = 2

nmodes = H1.shape[0]
print ('Number of modes:', nmodes)
print ('Number of modals:', nmodals)

3 mode term is missing. Setting 3 mode tensor to None.
Number of modes: 30
Number of modals: 5


## BLISS-Pauli

By setting syms=[] provides the 1-norm for the ordinary Pauli LCU. The symmetries A, B , C, etc can be understood from associated appendix of the paper.

In [5]:
H0_opt, H1_opt, H2_opt, H3_opt, bparams_dict = bp.optimize_bliss_pauli(obt=H1, tbt=H2, trbt=None, learning_rate=1e-1, syms=['A', 'B', 'C'], initial_guess=None, maxiter=50000, chunk_size=500, ret_params=True)

Symmetries considered for BLISS: ['A', 'B', 'C']
Starting optimization in 2-body space | Parameters strictly tracking 66 minimal degrees of freedom.


Iteration 0000 | 1-Norm Loss = 38042.307797
Iteration 0500 | 1-Norm Loss = 34860.044275
Iteration 1000 | 1-Norm Loss = 33212.965745
Iteration 1500 | 1-Norm Loss = 32503.338982
Iteration 2000 | 1-Norm Loss = 32064.993860
Iteration 2500 | 1-Norm Loss = 31722.267157
Iteration 3000 | 1-Norm Loss = 31384.711441
Iteration 3500 | 1-Norm Loss = 31053.710811
Iteration 4000 | 1-Norm Loss = 30748.633870
Iteration 4500 | 1-Norm Loss = 30442.364231
Iteration 5000 | 1-Norm Loss = 30137.974099
Iteration 5500 | 1-Norm Loss = 29832.447602
Iteration 6000 | 1-Norm Loss = 29581.817893
Iteration 6500 | 1-Norm Loss = 29334.530934
Iteration 7000 | 1-Norm Loss = 29082.314836
Iteration 7500 | 1-Norm Loss = 28826.210819
Iteration 8000 | 1-Norm Loss = 28567.300015
Iteration 8500 | 1-Norm Loss = 28306.537867
Iteration 9000 | 1-Norm Loss = 28049.139572
Iteration 9500 | 1-Norm Loss = 27823.713220
Iteration 10000 | 1-Norm Loss = 27607.597923
Iteration 10500 | 1-Norm Loss = 27400.223340
Iteration 11000 | 1-Norm Loss 

## Pauli QPE cost

In [46]:
#cutoff to prune LCU
cutoff = 0.012
bliss = False

if bliss == True:
  H0B, H1_main, H2_main, H3_main = bu.get_bliss_hamiltonian(H1, tbt=H2, trbt=None, params=bparams_dict)
  H2_nonsym_main = tu.unsymmetrize_tbt(H2_main)
  H3_nonsym_main = None
else:
  H1_main, H2_main, H3_main = H1, H2, None
  H2_nonsym_main, H3_nonsym_main = H2_nonsym, None

#Number of unqiue LCU terms
S = cu.get_Pauli_S(H1_main, H2_nonsym_main, H3_nonsym_main, cutoff)

#Pruning small coefficients
H1_main_approx, H2_nonsym_main_approx, H3_nonsym_main_approx, sparse_ten_error = cu.get_pruned_tensors_w_err(H1_main, H2_nonsym_main, H3_nonsym_main, mc_level, cutoff)
H2_main_approx = tu.symmetrize_tbt(H2_nonsym_main_approx) 
H3_main_approx = tu.symmetrize_trbt(H3_nonsym_main_approx)

print ("cutoff used = ", cutoff)
eps_ten = np.round(sparse_ten_error, 2)
rel_error = np.linalg.norm(H2_nonsym_main_approx - H2_nonsym_main)/np.linalg.norm(H2_nonsym_main)

one_norm = tu.opt_Pauli_LCU_1norm(obt=H1_main, tbt=H2_main_approx, trbt=H3_main_approx)
eps = 10

print ("Number of LCU coefficients to load: ", S)

QROAM_type = "dirty"

if QROAM_type == "dirty":
  dirty, clean = True, False
elif QROAM_type == "clean":
  dirty, clean = False, 605
elif (QROAM_type == "None") or (QROAM_type == "QROM") or (QROAM_type == "False"):
  dirty, clean = False, False
else:
  raise("Invalid QROAM type provided")

C_Pauli, N_Pauli = cu.get_Pauli_QPE_cost(one_norm, eps, nmodes, nmodals, S, mc_level, br = 20, aleph = 20, dirty = dirty, clean = clean)

print ("\nMolecule: ", mol_name)
print ("Modals per mode: ", nmodals)
print ("\nRelative error in tensor approximation:", np.round(rel_error,7), "cm-1")
print (f"Absolute error in tensor approximation: {eps_ten} cm-1")
print (f"Target accuracy in energy: {eps} cm-1")

cutoff used =  0.012
Number of LCU coefficients to load:  53884

QROAM ancilla cost: 15
QROAM inverse ancilla cost: 10

Cost of PREP:  5.43e+04
Cost of SEL:  7.39e+02
Cost of BE:  5.65e+04
LCU 1-norm:  1.05e+06 cm-1
# of walks:  1.65e+05
Toffoli cost of Sparse Pauli LCU:  9.33892e+09
Qubit cost of Sparse Pauli LCU: 333.0

Molecule:  pah1
Modals per mode:  5

Relative error in tensor approximation: 2.91e-05 cm-1
Absolute error in tensor approximation: 1.06 cm-1
Target accuracy in energy: 10 cm-1


In [54]:
import pennylane.labs.estimator_beta as qre_beta
import pennylane.estimator as qre

num_dirty = nmodes*nmodals + 20 + 20
print ("Number of available dirty qubits:", num_dirty)
print ("Number of bitstrings to load:", S)

SelCpy = qre_beta.SelectCopyQROM(num_bitstrings = S, size_bitstring = 20, available_dirty_aux = num_dirty)
print ("\nQROAM using Select Copy")
print(qre_beta.estimate(SelCpy))

SelSwp_Toffolis, SelSwp_clean, SelSwp_dirty = cu.get_QROAM_cost(S, 20, dirty=num_dirty)
print ("\nQROAM using Select Swap (my implementation)")
print ("Select Swap Toffolis:", SelSwp_Toffolis)
print ("Number of clean ancillas:", SelSwp_clean)
print ("Number of dirty ancillas:", SelSwp_dirty)
print ("Number of qubits to store S:", int(np.ceil(np.log2(S))))

qrom = qre.QROM(num_bitstrings = S, size_bitstring = 20, select_swap_depth = 8)
print ("\nQROAM using Select Swap (PennyLane implementation)")
print(qre.estimate(qrom))

Number of available dirty qubits: 190
Number of bitstrings to load: 53884

QROAM using Select Copy
--- Resources: ---
 Total wires: 234
   algorithmic wires: 36
   allocated wires: 198
     zero state: 198
     any state: 0
 Total gates : 7.076E+5
   'Toffoli': 8.641E+3,
   'CNOT': 6.624E+5,
   'X': 1.442E+4,
   'Hadamard': 2.214E+4

QROAM using Select Swap (my implementation)
Select Swap Toffolis: 14032
Number of clean ancillas: 13
Number of dirty ancillas: 140
Number of qubits to store S: 16

QROAM using Select Swap (PennyLane implementation)
--- Resources: ---
 Total wires: 188
   algorithmic wires: 36
   allocated wires: 152
     zero state: 152
     any state: 0
 Total gates : 1.187E+6
   'Toffoli': 1.403E+4,
   'CNOT': 1.106E+6,
   'X': 2.694E+4,
   'Hadamard': 4.044E+4
